In [1]:
from preprocessing import remove_slashes

with open('chembl_25_chemreps.txt', 'r') as txt_file:
    lines = txt_file.readlines()
txt_file.close()
lines = lines[:1000]
lines = [l.split('\t') for l in lines][1:]
smiles = [remove_slashes(l[1]) for l in lines]
print(smiles[:5])

['Cc1cc(cn1C)c2csc(N=C(N)N)n2', 'CC[C@H](C)[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@@H](N)CCSC)[C@@H](C)O)C(=O)NCC(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](Cc1c[nH]cn1)C(=O)N[C@@H](CC(=O)N)C(=O)NCC(=O)N[C@@H](C)C(=O)N[C@@H](C)C(=O)N[C@@H](CCC(=O)N)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N[C@@H](CCC(=O)N)C(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](CCCN=C(N)N)C(=O)NCC(=O)N[C@@H](CCC(=O)N)C(=O)N[C@@H](CC(C)C)C(=O)NCC(=O)N2CCC[C@H]2C(=O)N3CCC[C@H]3C(=O)NCC(=O)N[C@@H](CO)C(=O)N[C@@H](CCCN=C(N)N)C(=O)N', 'CCCC[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCCCN)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(=O)N)NC(=O)[C@H](CO)NC(=O)[C@H](Cc1c[nH]cn1)NC(=O)[C@H](C)NC(=O)[C@H](CCC(=O)N)NC(=O)[C@H](CCC(=O)N)NC(=O)[C@H](C)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CCC(=O)N)NC(=O)[C@@H]2CCCCNC(=O)CC[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC(=O)[C@H](CCC(=O)O)NC(=O)[C@H](CCCN=C(N)N)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc3c[nH]cn3)NC(=O)[C@H](N)Cc4ccccc4)C(C)C)C(=O)N[C@@H](CCCC)C(=O)N[C@@H](C)

In [2]:
from preprocessing import SMILESEncoder
from structs import SMILESDataset
from sklearn.model_selection import train_test_split

smiles_train, smiles_test = train_test_split(smiles, test_size=0.2, random_state=24)

encoder = SMILESEncoder(smiles_train, max_tokens=1000, n_merges=128)
print(encoder.vocab)

vectors_train = encoder.encode(smiles_train)
vectors_test = encoder.encode(smiles_test)

dataset_train = SMILESDataset(vectors_train)
dataset_test = SMILESDataset(vectors_test)

print(len(dataset_train), len(dataset_test))
print(dataset_train.shape, dataset_test.shape)

['CO', 'C(=', 'C1', 'NC(=O)[C@@H](', 'NC(=O)', 'c2', 'c', 's', 'c(', 'n', '2)', 'c3', 'cc(', 'O)', 'c4', '4)', '[C@@H]', '5', 'C', 'OC(=O)', 'c6', 'c7', '[C@@H](', '[C@H](', '8', 'c1', ')', '9', 'C(=O)N', '5)', 'O', '[C@H]', '%10', '[C@]', '(C)', '(O)', '[C@H](C)', 'N', 'C)', 'C(=O)O', 'Cc', '%1', '1', 'cc', '1)', '6', '2', '3', 'CCN(', 'CC', '3)', '[C@@H](C)O)', '[C@H]1', '[C@@H]2', '[C@H](O)', 'C=', 'C2', '[', 'B', 'r', '-]', '.', '+]', '(=O)', 'ccc(', '[n', '(', 'CCCC', 'ccc(cc', '[C@@H](O)', 'N)', 'CCC', 'C(=O)N[C@@H](CCCNC(=N)N)', 'C(=O)N[C@@H](C)', 'C(=O)N[C@@H](CCCCN)', 'C(=O)N[C@@H](', 'CO)', '[C@@H]1', 'O[C@H](', '[C@H]2', 'O[C@@H]', 'C[C@H](', '[C@H]3', '4', '[C@@H](O)[C@H](O)[C@H]', '6)', 'cn', 'C(=O)', 'ccc', 'C(C)(C)', '=', 'CC(C)', 'Cc2', 'ccc(O)cc', 'ccccc', 'COc1', 'Cc3', 'c5', 'CCCNC(=', 'C(', 'F)', 'F', '#', 'Cl)', 'CCCN', '[nH]', '7', 'S', 'ccccc2)', '=O)', 'CC=', 'C3', '[C@@]', 'CC[C@]', 'Cl', 'l', '=C(', 'N)N)', '[C@@H](C)', 'CC)', 'C(=O)N[C@@H](C', 'C(C)C)', 'C(=O

In [5]:
from vae2 import train_vae, VAE

vae = VAE(
    (dataset_train.shape[1], dataset_train.shape[2]),
    512, 16
)

vae = train_vae(vae, dataset_train, epochs=200, verbose=1)

Epoch: 0 | Train loss: nan
Epoch: 1 | Train loss: nan


KeyboardInterrupt: 

In [4]:
recon_train_vectors = vae(dataset_train.X).detach().numpy()
print(recon_train_vectors.shape)

(368, 42, 43)


In [5]:
from preprocessing import floats_to_onehot

recon_vect_onehot = floats_to_onehot(recon_train_vectors)
print(recon_vect_onehot.shape)

(368, 42, 43)


In [6]:
recon_smiles = encoder.decode(recon_vect_onehot)

for i in range(10):
    print(smiles_train[i], recon_smiles[i])

CC(C)CC(C)(C)C CCCCCCCCCC(=O)O
CC(=O)OCC1CCCO1 CCCCCCCC=CCC(=O)
CCOC(=O)C(=O)OCC CCCCOCC
CCCCCC(=O)CC CCCCCCCC=CCC
C1CCC(CC1)O CCCCCCCC=CCC
C=COCC1CCC(CC1)COC=C CCCC=C(C(C1
CCCCCCCCCC(=O)OC(C)C CCCCCCCCCC(=O)O
CCCO CCCCCCCCCC(=O)O
CCCCCCCCCCCCCCCCCCCC(=O)OC CCCC=CCCCCC
CCC(C)CC(C)C CCCCCC(=O)O
